# 🎙️ Teaching Machines to Hear Ghanaian English — Live Demo
**NAIL Seminar · Dr. Amina Salifu · Research Lead, Northern AI Lab**

This notebook rebuilds the core pipeline from the PhD thesis
*"Enhancing Speech Recognition for Low-Resource Accents through CNN–Transformer Models and Shared Feature Extraction"* (KNUST, 2025) in five short, self-contained parts:

| Part | What we do | Thesis link |
|---|---|---|
| 1 | Turn sound into pictures (log-Mel spectrograms) | Preprocessing pipeline (all studies) |
| 2 | Mask the spectrogram + mLFR frame reduction | Study 2 — CNN-MSR |
| 3 | Train a mini CNN accent classifier | Studies 1 & 2 |
| 4 | Map the accent space (t-SNE + cosine similarity) | Study 2 — accent similarity |
| 5 | Transcribe + score with WER/CER | Study 3 — Ghanaian ASR |

**Live-demo safety:** every part runs fully offline on CPU. If you point `DATA_DIR` at the
[Accent Classification Dataset (Ghana)](https://www.kaggle.com/datasets/responsibleailab/accent-classification-dataset-ghana)
it uses real Akan/Dagomba/Ga/Ewe audio; otherwise it synthesises four distinct *pseudo-accents* so nothing can break on stage. Part 5 uses Whisper only if it's installed — otherwise it falls back to pre-recorded transcripts.


In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────
# pip install -r requirements.txt   (numpy librosa soundfile matplotlib scikit-learn torch jiwer)
import os, glob, warnings, random
import numpy as np
import librosa, librosa.display
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
random.seed(0); np.random.seed(0)

SR          = 16_000      # thesis: audio sampled at 16 kHz
N_MELS      = 64          # thesis: 64 Mel bands
WIN_MS, HOP_MS = 25, 10   # thesis: 25 ms window, 10 ms hop
N_FFT       = int(SR * WIN_MS / 1000)
HOP         = int(SR * HOP_MS / 1000)
SEG_FRAMES  = 100         # thesis: 1-second segments = 100 frames

# Point this at the Kaggle Ghana dataset (folders per accent) to use real audio:
DATA_DIR = "data"         # e.g. data/Akan/*.wav, data/Ewe/*.wav ...
ACCENTS  = ["Akan", "Dagomba", "Ga", "Ewe"]
print("Config OK — sample rate", SR, "Hz,", N_MELS, "Mel bands")

## Part 0 · Get audio (real if available, synthetic otherwise)
If the Ghana dataset is on disk we load real clips. If not, we synthesise four **pseudo-accents** —
each with its own pitch level, formant colouring, and speech rhythm — mimicking how real accents
differ systematically in *prosody* (melody/rhythm) and *acoustics* (spectral colour).
That systematic difference is exactly what the thesis models learn.

In [ ]:
def synth_voice(accent_id, dur=2.0, sr=SR):
    """Synthesise a vowel-like 'utterance' whose pitch, formants and rhythm
    depend on the accent id — a stand-in for real accent structure."""
    t = np.linspace(0, dur, int(sr * dur), endpoint=False)
    profiles = {  # f0 base, formant pair (Hz), syllable rate (Hz)
        0: (110, (700, 1200), 3.0),   # pseudo-Akan
        1: (135, (550, 1650), 4.0),   # pseudo-Dagomba
        2: (150, (800, 1400), 3.5),   # pseudo-Ga
        3: (125, (600, 1900), 4.5),   # pseudo-Ewe
    }
    f0, (F1, F2), rate = profiles[accent_id]
    f0 = f0 * (1 + 0.06 * np.random.randn())                    # speaker variation
    vibrato = 1 + 0.02 * np.sin(2 * np.pi * 5 * t)
    src = np.sin(2 * np.pi * f0 * vibrato * t)                  # glottal source
    src += 0.5 * np.sin(2 * np.pi * 2 * f0 * t) + 0.25 * np.sin(2 * np.pi * 3 * f0 * t)
    formants = (np.sin(2 * np.pi * F1 * t) + 0.7 * np.sin(2 * np.pi * F2 * t))
    voice = src * (0.6 + 0.4 * formants / np.abs(formants).max())
    envelope = 0.55 + 0.45 * np.clip(np.sin(2 * np.pi * rate * t), 0, None) ** 0.5  # syllable rhythm
    y = voice * envelope + 0.01 * np.random.randn(len(t))
    return (y / np.abs(y).max()).astype(np.float32)

def load_clips(n_per_accent=12):
    clips, labels, source = [], [], "synthetic pseudo-accents"
    for i, acc in enumerate(ACCENTS):
        files = glob.glob(os.path.join(DATA_DIR, acc, "*.wav")) + \
                glob.glob(os.path.join(DATA_DIR, acc, "*.mp3"))
        if files:
            source = f"real audio from {DATA_DIR}/"
            for f in sorted(files)[:n_per_accent]:
                y, _ = librosa.load(f, sr=SR, duration=3.0)
                clips.append(y); labels.append(i)
        else:
            for _ in range(n_per_accent):
                clips.append(synth_voice(i)); labels.append(i)
    return clips, np.array(labels), source

clips, labels, source = load_clips()
print(f"Loaded {len(clips)} clips across {len(ACCENTS)} accents — using {source}")

## Part 1 · How a machine hears: waveform → log-Mel spectrogram
The ear-inspired recipe used everywhere in the thesis: STFT with a 25 ms window / 10 ms hop,
then 64 Mel bands. The **spectrogram is the model's view of your voice** — accents live in its patterns.

In [ ]:
def logmel(y):
    S = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT,
                                       hop_length=HOP, n_mels=N_MELS)
    return librosa.power_to_db(S, ref=np.max)

fig, axes = plt.subplots(2, 2, figsize=(13, 6))
y = clips[0]
axes[0,0].plot(np.arange(len(y))/SR, y, lw=0.4, color="#0B6E5D")
axes[0,0].set(title=f"1. Raw waveform ({ACCENTS[labels[0]]})", xlabel="time (s)")
D = librosa.amplitude_to_db(np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP)), ref=np.max)
librosa.display.specshow(D, sr=SR, hop_length=HOP, x_axis="time", y_axis="hz", ax=axes[0,1], cmap="magma")
axes[0,1].set(title="2. STFT spectrogram (linear frequency)")
M = logmel(y)
librosa.display.specshow(M, sr=SR, hop_length=HOP, x_axis="time", y_axis="mel", ax=axes[1,0], cmap="magma")
axes[1,0].set(title="3. Log-Mel spectrogram — 64 bands (what the CNN sees)")
# compare two accents side by side
M2 = logmel(clips[-1])
librosa.display.specshow(M2, sr=SR, hop_length=HOP, x_axis="time", y_axis="mel", ax=axes[1,1], cmap="magma")
axes[1,1].set(title=f"Same view, different accent ({ACCENTS[labels[-1]]}) — spot the pattern shift")
plt.tight_layout(); plt.show()
print("Talking point: pitch bands sit at different heights, energy pulses at different rhythms —")
print("systematic differences a model can learn.")

## Part 2 · Study 2's engine: masking + mLFR
**Masked Spectrogram Reconstruction (MSR):** hide random time/frequency patches and train the model
to repaint them. To succeed, it must internalise the accent's structure — self-supervised learning that
needs **no labels**, perfect for low-resource settings.

**mLFR (much Lower Frame Rate):** stack neighbouring frames and downsample — the same information in a
shorter sequence, so training is cheaper. *Efficiency is a design goal, not an afterthought.*

In [ ]:
def mask_spectrogram(M, n_time=2, n_freq=2, max_t=15, max_f=10):
    Mm = M.copy(); fill = M.min()
    for _ in range(n_time):
        t0 = np.random.randint(0, max(1, Mm.shape[1]-max_t)); Mm[:, t0:t0+np.random.randint(5, max_t)] = fill
    for _ in range(n_freq):
        f0 = np.random.randint(0, max(1, Mm.shape[0]-max_f)); Mm[f0:f0+np.random.randint(3, max_f), :] = fill
    return Mm

def mlfr(M, stack=3, stride=3):
    """Stack `stack` consecutive frames, keep every `stride`-th — sequence gets ~3x shorter."""
    T = (M.shape[1] - stack) // stride + 1
    return np.stack([M[:, i*stride:i*stride+stack].reshape(-1) for i in range(T)], axis=1)

M = logmel(clips[0]); Mm = mask_spectrogram(M); Ml = mlfr(M)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, S, ttl in zip(axes, [M, Mm, Ml],
        [f"Original ({M.shape[0]}x{M.shape[1]})",
         "Masked — model must repaint the gaps",
         f"mLFR stacked ({Ml.shape[0]}x{Ml.shape[1]}) — 3x shorter sequence"]):
    ax.imshow(S, aspect="auto", origin="lower", cmap="magma"); ax.set_title(ttl, fontsize=10)
plt.tight_layout(); plt.show()
print("Thesis result: MSR pre-training lifted accent accuracy from 81.9% (plain CNN) to 90.7%,")
print("reaching >90% validation accuracy in under 50 epochs.")

## Part 3 · A mini accent classifier (CNN)
A scaled-down version of the thesis CNN, trained on 1-second log-Mel segments (100 frames).
On stage this trains in ~30 seconds on CPU. With the real Kaggle dataset and the full
architecture, this same recipe reached **90.71%** combined accuracy (native 100%, cross-native 99.31%).

In [ ]:
import torch, torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

def segments(y, seg=SEG_FRAMES):
    M = logmel(y)
    M = (M - M.mean()) / (M.std() + 1e-6)
    out = []
    for s in range(0, M.shape[1] - seg + 1, seg):
        out.append(M[:, s:s+seg])
    if not out:                                # pad short clips (thesis: pad/truncate)
        pad = np.zeros((N_MELS, seg)); pad[:, :M.shape[1]] = M; out = [pad]
    return out

X, Y = [], []
for y_, l in zip(clips, labels):
    for seg in segments(y_):
        X.append(seg); Y.append(l)
X = torch.tensor(np.array(X), dtype=torch.float32).unsqueeze(1)   # (N,1,64,100)
Y = torch.tensor(np.array(Y))
Xtr, Xte, Ytr, Yte = train_test_split(X, Y, test_size=0.3, stratify=Y, random_state=0)
print("Training segments:", len(Xtr), "· Test segments:", len(Xte))

class MiniAccentCNN(nn.Module):
    def __init__(self, n_classes=len(ACCENTS)):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1))
        self.head = nn.Linear(64, n_classes)
    def embed(self, x):  return self.net(x).flatten(1)      # accent embedding (Part 4)
    def forward(self, x): return self.head(self.embed(x))

model = MiniAccentCNN(); opt = torch.optim.Adam(model.parameters(), lr=2e-3)
lossf = nn.CrossEntropyLoss()
for epoch in range(15):
    model.train(); opt.zero_grad()
    loss = lossf(model(Xtr), Ytr); loss.backward(); opt.step()
    if epoch % 5 == 4:
        model.eval()
        with torch.no_grad():
            acc = accuracy_score(Yte, model(Xte).argmax(1))
        print(f"epoch {epoch+1:2d} · loss {loss.item():.3f} · test acc {acc:.2%}")

model.eval()
with torch.no_grad(): pred = model(Xte).argmax(1)
ConfusionMatrixDisplay(confusion_matrix(Yte, pred), display_labels=ACCENTS).plot(cmap="Greens")
plt.title("Mini CNN — accent confusion matrix"); plt.show()

## Part 4 · Mapping the accent space
The thesis used **t-SNE / UMAP** to visualise accent embeddings and **cosine similarity** to quantify
accent relationships. Finding: native accents form tight, separate islands; Ghanaian accents overlap —
real phonetic diversity, and shared structure that shared-feature methods exploit.

In [ ]:
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

with torch.no_grad():
    E = model.embed(X).numpy()
Z = TSNE(n_components=2, perplexity=12, random_state=0).fit_transform(E)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
colors = ["#0B6E5D", "#6FBF57", "#E8963C", "#B85042"]
for i, acc in enumerate(ACCENTS):
    m = Y.numpy() == i
    axes[0].scatter(Z[m,0], Z[m,1], s=32, alpha=0.8, color=colors[i], label=acc)
axes[0].legend(); axes[0].set_title("t-SNE of CNN accent embeddings")

centroids = np.stack([E[Y.numpy()==i].mean(0) for i in range(len(ACCENTS))])
S = cosine_similarity(centroids)
im = axes[1].imshow(S, cmap="Greens", vmin=0, vmax=1)
axes[1].set_xticks(range(len(ACCENTS)), ACCENTS); axes[1].set_yticks(range(len(ACCENTS)), ACCENTS)
for i in range(len(ACCENTS)):
    for j in range(len(ACCENTS)):
        axes[1].text(j, i, f"{S[i,j]:.2f}", ha="center", va="center",
                     color="white" if S[i,j] > .6 else "#24312D")
axes[1].set_title("Cosine similarity between accent centroids")
plt.colorbar(im, ax=axes[1]); plt.tight_layout(); plt.show()

## Part 5 · Study 3: does it transcribe? WER & CER
The metric that users feel. Baseline Whisper on Ghanaian accents (thesis Table 3.26) vs the
fine-tuned pipeline (Table 3.27):

| Accent | WER before | WER after | CER before | CER after |
|---|---|---|---|---|
| Akan | 33.97% | **22.84%** | 23.73% | **15.36%** |
| Dagomba | 34.20% | **30.70%** | 24.90% | **27.02%** |
| Ewe | 49.20% | **33.63%** | 37.35% | **24.48%** |

Below we compute WER/CER live. If `openai-whisper` is installed and you have a real clip, it transcribes it;
otherwise we score example transcript pairs that illustrate typical accent-driven ASR errors.

In [ ]:
from jiwer import wer, cer

reference = "the market at kejetia opens very early in the morning"
# Typical baseline vs improved hypotheses (accent-driven substitutions on names & vowels):
hyp_baseline  = "the market at cage tia opens fairy early in the morning"
hyp_finetuned = "the market at kejetia opens very early in the morning"

USE_WHISPER = False   # flip to True live if whisper is installed and you have a wav
if USE_WHISPER:
    import whisper
    wmodel = whisper.load_model("base")
    hyp_baseline = wmodel.transcribe("data/Akan/example.wav")["text"].lower().strip()

for name, hyp in [("Baseline", hyp_baseline), ("Fine-tuned", hyp_finetuned)]:
    print(f"{name:11s} | WER {wer(reference, hyp):.2%} · CER {cer(reference, hyp):.2%} | '{hyp}'")

print()
print("Same story as the thesis: accent-aware fine-tuning turns systematic mishearings")
print("('kejetia' → 'cage tia') into correct transcriptions — WER for Ewe fell 49% → 34%.")

## Wrap-up — reproduce, then push past it
1. **Download** the dataset: [kaggle.com/datasets/responsibleailab/accent-classification-dataset-ghana](https://www.kaggle.com/datasets/responsibleailab/accent-classification-dataset-ghana) and set `DATA_DIR`.
2. **Scale up** Part 3: more layers, MSR pre-training (Part 2's masking as a reconstruction objective), attention pooling.
3. **Fine-tune** Whisper (or wav2vec2) on accent-partitioned data and re-measure Part 5.
4. **Extend**: more Ghanaian accents, spontaneous speech, code-switching.

*Medaase — and reach out to NAIL (@NorthernGhanaAILab) to collaborate.*